Day 5, Topic 4: np.einsum – Einstein Summation for Matrix Products, Traces, Batches

In [1]:
#Example 1: Element‑wise Multiplication
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

#Numpy edition
np_result = a * b

#einsum edition
einsum_result = np.einsum('i,i->i', a, b)

print(einsum_result)

[ 4 10 18]


In [2]:
#Example 2: Vector Dot Product
einsum_result = np.einsum('i, i->', a, b)
print(einsum_result)

32


In [4]:
#Example 3: Matrix Transpose
M = np.arange(12).reshape(4, 3)

#Numpy edition
result = M.T

#einsum edition
einsum_result = np.einsum('ij->ji', M)

print(M)
print(einsum_result)

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
[[ 0  3  6  9]
 [ 1  4  7 10]
 [ 2  5  8 11]]


In [6]:
#Example 4: Sum Over Rows or Columns
arr = np.arange(6).reshape(2, 3)

col = np.einsum('ij->j', arr)
print(col)

row = np.einsum('ij->i', arr)
print(row)

[3 5 7]
[ 3 12]


In [7]:
#Example 5: Trace (Sum of Diagonal)
M = np.arange(9).reshape(3, 3)

trace = np.einsum('ii->', M)
print(trace)

12


In [10]:
#Part 3: Matrix Multiplication
A = np.arange(6).reshape(2, 3)
B = np.arange(12).reshape(3, 4)

# NumPy way
np_result = A @ B   # or np.dot(A, B)

# einsum way
einsum_result = np.einsum('ik,kj->ij', A, B)

print(einsum_result)
# [[20 23 26 29]
#  [56 68 80 92]]

[[20 23 26 29]
 [56 68 80 92]]


In [11]:
#Part 4: Batch Matrix Multiplication (3D Tensors)
batch_size = 10
A = np.random.rand(batch_size, 2, 3)
B = np.random.rand(batch_size, 3, 4)

#Numpy way
result = np.matmul(A, B)

#einsum way
result_1 = np.einsum('bij,bjk->bik', A, B)

print(result_1.shape)

(10, 2, 4)


In [12]:
#Example: Outer Product
a = np.array([1, 2, 3])
b = np.array([4, 5])

result = np.einsum('i,j->ij', a, b)

print(result)

[[ 4  5]
 [ 8 10]
 [12 15]]


In [13]:
#Example: Diagonal Extraction as Vector
M = np.arange(9).reshape(3, 3)

result = np.einsum('ii->i', M)
print(result)

[0 4 8]


In [17]:
import numpy as np

X = np.random.rand(5, 3)  # 5 points in 3D
Y = np.random.rand(4, 3)  # 4 points in 3D

# Squared distances: ||X_i - Y_j||^2 = sum_k (X_ik - Y_jk)^2
# Expand to: sum_k X_ik^2 + sum_k Y_jk^2 - 2 * sum_k X_ik Y_jk

X_sq = np.einsum('ij,ij->i', X, X)[:, np.newaxis]  # Shape: (5, 1)
Y_sq = np.einsum('ij,ij->i', Y, Y)[np.newaxis, :]  # Shape: (1, 4)
XY = np.einsum('ik,jk->ij', X, Y)                  # Shape: (5, 4)

dist_sq = X_sq + Y_sq - 2 * XY

In [18]:
A = np.random.rand(100, 200, 50)
B = np.random.rand(50, 300)
C = np.random.rand(300, 400)

path_info = np.einsum_path('ijk,kl,lm->ijm', A, B, C, optimize='optimal')
print(path_info[0])


result = np.einsum('ijk,kl,lm->ijm', A, B, C, optimize='optimal')

['einsum_path', (1, 2), (0, 1)]


In [19]:
#Task: Given the following arrays, use einsum to perform the specified operations.

In [20]:
A = np.random.rand(4, 3, 2)
B = np.random.rand(2, 5)
C = np.random.rand(4, 3)
D = np.random.rand(3, 6)

In [21]:
#Multiply A (4×3×2) with B (2×5) along the last axis of A and first axis of B, resulting in shape (4, 3, 5).
result = np.einsum('ijk,kl->ijl', A, B)
print(result.shape)

(4, 3, 5)


In [22]:
#compute the sum of the element‑wise product of A's first two dimensions for each element along the last axis? 
#Let's do: Compute sum_{i,j} A[i,j,k] * C[i,j] resulting in shape (2,).
#(Hint: Use indices 'ijk,ij->k'.)
result = np.einsum('ijk,ij->k', A, C)
print(result.shape)

(2,)


In [23]:
#Perform batch outer product: For each of the 4 batches, compute the outer product of the 3‑element 
#vector C[i] with the 6‑element vector from D? Actually C shape is (4,3) and D is (3,6). Compute C @ D using einsum to get (4,6).
result = np.einsum('ij, jk->ik', C, D)
print(result.shape)

(4, 6)


In [24]:
import numpy as np

# 4 points in 3D space
C = np.random.rand(4, 3)

# 1. Squared norms: ||C_i||^2 and ||C_j||^2
# 'id,id->i' squares each element and sums across the 3D coordinates (axis 'd')
# This gives a 1D array of shape (4,) containing the squared magnitude of each point.
C_sq = np.einsum('id,id->i', C, C)

# 2. Dot products: C_i * C_j
# 'id,jd->ij' computes the dot product between every point 'i' and every point 'j'
# This results in a (4, 4) matrix.
C_dot = np.einsum('id,jd->ij', C, C)

# 3. Combine using broadcasting
# We reshape C_sq to (4, 1) and (1, 4) so they broadcast over the (4, 4) grid.
dist_sq = C_sq[:, np.newaxis] + C_sq[np.newaxis, :] - 2 * C_dot

print("Shape of distance matrix:", dist_sq.shape)
print("\nSquared Distance Matrix:\n", dist_sq)

Shape of distance matrix: (4, 4)

Squared Distance Matrix:
 [[0.         1.06986769 1.00507921 1.02454047]
 [1.06986769 0.         0.01273978 0.62374455]
 [1.00507921 0.01273978 0.         0.72956993]
 [1.02454047 0.62374455 0.72956993 0.        ]]


In [ ]:
#Day 5, Topic 5: Stride Tricks – sliding_window_view for Rolling Operations

In [31]:
#1D Sliding Window
from numpy.lib.stride_tricks import sliding_window_view

a = np.arange(10)
windows = sliding_window_view(a, window_shape=4)

print(windows.shape)
a[1] = 999
print(windows)

(7, 4)
[[  0 999   2   3]
 [999   2   3   4]
 [  2   3   4   5]
 [  3   4   5   6]
 [  4   5   6   7]
 [  5   6   7   8]
 [  6   7   8   9]]


In [37]:
#2D Sliding Window (Image Patches)
b = np.arange(25).reshape(5, 5)

patches = sliding_window_view(b, window_shape=(3, 3))

print(patches.shape)

for i in range(patches.shape[0]):
    for j in range(patches.shape[1]):
        print(f"Patch ({i}, {j}):\n", patches[i, j])

(3, 3, 3, 3)
Patch (0, 0):
 [[ 0  1  2]
 [ 5  6  7]
 [10 11 12]]
Patch (0, 1):
 [[ 1  2  3]
 [ 6  7  8]
 [11 12 13]]
Patch (0, 2):
 [[ 2  3  4]
 [ 7  8  9]
 [12 13 14]]
Patch (1, 0):
 [[ 5  6  7]
 [10 11 12]
 [15 16 17]]
Patch (1, 1):
 [[ 6  7  8]
 [11 12 13]
 [16 17 18]]
Patch (1, 2):
 [[ 7  8  9]
 [12 13 14]
 [17 18 19]]
Patch (2, 0):
 [[10 11 12]
 [15 16 17]
 [20 21 22]]
Patch (2, 1):
 [[11 12 13]
 [16 17 18]
 [21 22 23]]
Patch (2, 2):
 [[12 13 14]
 [17 18 19]
 [22 23 24]]


In [41]:
#Specifying Axes
arr_2d = np.arange(12).reshape(3, 4)

windows = sliding_window_view(arr_2d, window_shape=3, axis=1)
print(windows.shape)

(3, 2, 3)


In [43]:
#Multi‑Dimensional Windows on Multi‑Axes
arr_3d = np.arange(60).reshape(3, 4, 5)

windows = sliding_window_view(arr_3d, window_shape=(2, 2), axis=(1, 2))
print(windows.shape)

(3, 3, 4, 2, 2)


In [44]:
#Moving Average
arr = np.random.rand(1000)
window = 50

avg = sliding_window_view(arr, window)
monthly_avg = avg.mean(axis=1)

print(monthly_avg.shape)

(951,)


In [46]:
#Image Convolution
images = np.random.randint(0, 256, (512, 512), dtype=np.uint8)
kernel_size = 3

patches = sliding_window_view(images, (kernel_size, kernel_size))
print(patches.shape)

#Filtering patches
patches_mean = patches.mean(axis=(2, 3))
print(patches_mean.shape)

(510, 510, 3, 3)
(510, 510)


In [47]:
#Task: You have a 2D grayscale image of shape (1000, 1000) represented as a float32 array. Use sliding_window_view to:
image_2d = np.random.rand(1000, 1000).astype(np.float32)

In [48]:
#Extract all 5×5 patches with stride 1 (overlapping). What is the shape of the resulting view?
patch_size = 5
patches = sliding_window_view(image_2d, (patch_size, patch_size))
print(patches.shape)

(996, 996, 5, 5)


In [49]:
#Apply a 5×5 Gaussian blur approximation by replacing each pixel with the mean of its 5×5 neighborhood. 
#Use the patches to compute the blurred image without loops.
#Verify that the blurred image has shape (996, 996).
blurred_image = patches.mean(axis=(-2, -1))
print(blurred_image.shape)

(996, 996)


In [50]:
#Instead of stride 1, 
#extract non‑overlapping 5×5 patches (stride 5). Hint: First slice the array with step 5, then apply sliding_window_view with window size 1?
non_overlapping_image = patches[::5, ::5]
print(non_overlapping_image.shape)

(200, 200, 5, 5)
